# Heimdall Face Recognition Model Training

**Target:** 97% accuracy on noisy images

**Current:** 13.3% on noise (σ=30)

---

## Setup Instructions
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Run all cells in order
3. Download the trained model at the end

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch torchvision facenet-pytorch albumentations opencv-python-headless PyYAML tqdm gdown

In [ ]:
# Verify installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Download CASIA-WebFace Dataset (500K images)

This is a much larger dataset than LFW and will produce better noise robustness.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

# Create directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed/train', exist_ok=True)
os.makedirs('data/processed/val', exist_ok=True)
os.makedirs('data/processed/test', exist_ok=True)
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('models/final', exist_ok=True)

# Try multiple download methods
DATA_DIR = Path("data/raw/faces")
DATA_DIR.mkdir(parents=True, exist_ok=True)

download_success = False

# Method 1: Try CASIA-WebFace from Google Drive
print("="*50)
print("Attempting to download CASIA-WebFace (500K images)...")
print("="*50)

!pip install -q gdown

try:
    import gdown
    # Try multiple Google Drive sources
    sources = [
        ("1Of_EVz-yHV7QVWQGihYfvtny9Ne8qXVz", "CASIA original"),
        ("1KxNCrXzln0lal3N4JiYl9cFOIhT78y1l", "CASIA mirror"),
    ]
    
    for file_id, name in sources:
        print(f"\nTrying {name}...")
        zip_path = "data/raw/casia.zip"
        try:
            url = f"https://drive.google.com/uc?id={file_id}"
            gdown.download(url, zip_path, quiet=False)
            
            if os.path.exists(zip_path) and os.path.getsize(zip_path) > 500_000_000:  # >500MB
                print(f"Download successful! Size: {os.path.getsize(zip_path)/1e9:.2f} GB")
                print("Extracting (this takes several minutes)...")
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall("data/raw")
                
                # Find extracted folder
                for folder in ["CASIA-WebFace", "casia-webface", "webface"]:
                    if (Path("data/raw") / folder).exists():
                        shutil.move(str(Path("data/raw") / folder), str(DATA_DIR))
                        download_success = True
                        break
                
                if download_success:
                    break
        except Exception as e:
            print(f"Failed: {e}")
            if os.path.exists(zip_path):
                os.remove(zip_path)
except Exception as e:
    print(f"gdown failed: {e}")

# Method 2: Fallback to LFW via sklearn
if not download_success:
    print("\n" + "="*50)
    print("CASIA download failed. Using LFW dataset instead.")
    print("="*50)
    
    from sklearn.datasets import fetch_lfw_people
    print("Downloading LFW via sklearn...")
    lfw = fetch_lfw_people(min_faces_per_person=1, resize=1.0, download_if_missing=True)
    print(f"Downloaded {len(lfw.images)} images")
    
    # Get sklearn data location
    import sklearn
    from sklearn.datasets import get_data_home
    lfw_dir = Path(get_data_home()) / "lfw_home" / "lfw_funneled"
    
    if lfw_dir.exists():
        # Copy to our data directory
        print(f"Copying from {lfw_dir}...")
        for person_dir in lfw_dir.iterdir():
            if person_dir.is_dir():
                dest = DATA_DIR / person_dir.name
                if not dest.exists():
                    shutil.copytree(person_dir, dest)
        download_success = True
        print("LFW data copied successfully!")

# Method 3: Direct download LFW if sklearn fails
if not download_success:
    print("\nTrying direct LFW download...")
    lfw_url = "http://vis-www.cs.umass.edu/lfw/lfw-funneled.tgz"
    !wget -q --show-progress -O data/raw/lfw.tgz "$lfw_url" || curl -L -o data/raw/lfw.tgz "$lfw_url"
    
    if os.path.exists("data/raw/lfw.tgz"):
        import tarfile
        with tarfile.open("data/raw/lfw.tgz", "r:gz") as tar:
            tar.extractall("data/raw")
        if (Path("data/raw") / "lfw_funneled").exists():
            shutil.move("data/raw/lfw_funneled", str(DATA_DIR))
            download_success = True

# Verify
if DATA_DIR.exists():
    identities = [d for d in DATA_DIR.iterdir() if d.is_dir()]
    total_images = sum(len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png"))) for d in identities[:100])
    print(f"\n{'='*50}")
    print(f"Dataset ready at: {DATA_DIR}")
    print(f"Identities: {len(identities)}")
    print(f"Sample image count (first 100 identities): {total_images}")
    print(f"{'='*50}")
else:
    print("\nERROR: No dataset available!")
    print("Please manually download CASIA-WebFace and upload to Colab.")

In [ ]:
# Prepare train/val/test splits
import random
import shutil
from tqdm import tqdm

# Use the data directory from previous cell
raw_dir = Path('data/raw/faces')

if not raw_dir.exists() or not any(raw_dir.iterdir()):
    raise FileNotFoundError(f"No data found at {raw_dir}. Please re-run the download cell.")

print(f"Using data from: {raw_dir}")

train_dir = Path('data/processed/train')
val_dir = Path('data/processed/val')
test_dir = Path('data/processed/test')

# Clear existing processed data
for d in [train_dir, val_dir, test_dir]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

# Get all identity directories
identity_dirs = [d for d in raw_dir.iterdir() if d.is_dir()]
print(f"Total identities: {len(identity_dirs)}")

total_train = 0
total_val = 0
total_test = 0
valid_identities = 0

# Need at least 5 images per identity for good training
MIN_IMAGES = 5

for identity_dir in tqdm(identity_dirs, desc="Processing"):
    images = list(identity_dir.glob('*.jpg')) + list(identity_dir.glob('*.png')) + list(identity_dir.glob('*.JPG')) + list(identity_dir.glob('*.jpeg'))
    
    if len(images) < MIN_IMAGES:
        continue
    
    valid_identities += 1
    random.shuffle(images)
    
    # 80/10/10 split
    n_train = max(3, int(len(images) * 0.8))
    n_val = max(1, (len(images) - n_train) // 2)
    
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train+n_val]
    test_imgs = images[n_train+n_val:]
    
    # Copy files
    for split_dir, split_imgs in [(train_dir, train_imgs), (val_dir, val_imgs), (test_dir, test_imgs)]:
        if not split_imgs:
            continue
        dest_dir = split_dir / identity_dir.name
        dest_dir.mkdir(exist_ok=True)
        for img in split_imgs:
            shutil.copy2(img, dest_dir / img.name)
    
    total_train += len(train_imgs)
    total_val += len(val_imgs)
    total_test += len(test_imgs)

print(f"\nDataset prepared:")
print(f"  Valid identities (5+ images): {valid_identities}")
print(f"  Training: {total_train:,} images")
print(f"  Validation: {total_val:,} images")
print(f"  Test: {total_test:,} images")

if valid_identities < 50:
    print("\nWARNING: Very few identities with 5+ images!")
    print("Consider lowering MIN_IMAGES to 2 or 3 for better results.")

## 2. Define ArcFace Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ArcFaceLoss(nn.Module):
    """ArcFace Loss with Additive Angular Margin."""
    
    def __init__(self, embedding_dim=512, num_classes=1000, scale=64.0, margin=0.5):
        super().__init__()
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin
    
    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weight = F.normalize(self.weight, p=2, dim=1)
        
        cosine = F.linear(embeddings, weight)
        cosine = torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7)
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        
        return F.cross_entropy(output, labels)
    
    def get_cosine_similarity(self, embeddings):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weight = F.normalize(self.weight, p=2, dim=1)
        return F.linear(embeddings, weight)

print("ArcFace Loss defined!")

## 3. Define Dataset with Heavy Augmentation

In [ ]:
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_training_augmentation():
    """Heavy augmentation for noise robustness."""
    return A.Compose([
        # CRITICAL: Heavy noise augmentation
        A.OneOf([
            A.GaussNoise(var_limit=(100, 2500), p=1.0),
            A.ISONoise(intensity=(0.1, 0.5), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.8, 1.2), p=1.0),
        ], p=0.7),
        
        # Rotation
        A.Rotate(limit=45, border_mode=cv2.BORDER_REPLICATE, p=0.5),
        
        # Lighting
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
            A.RandomGamma(gamma_limit=(80, 120), p=1.0),
            A.CLAHE(clip_limit=4.0, p=1.0),
        ], p=0.6),
        
        # Grayscale
        A.ToGray(p=0.2),
        
        # Quality degradation
        A.OneOf([
            A.ImageCompression(quality_lower=30, quality_upper=100, p=1.0),
            A.Downscale(scale_min=0.5, scale_max=0.9, p=1.0),
            A.GaussianBlur(blur_limit=7, p=1.0),
            A.MotionBlur(blur_limit=7, p=1.0),
        ], p=0.5),
        
        # Occlusion
        A.CoarseDropout(max_holes=3, max_height=30, max_width=30, p=0.3),
        
        # Normalize
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ToTensorV2()
    ])

def get_validation_transform():
    return A.Compose([
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ToTensorV2()
    ])

class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None, min_samples=5):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}
        
        idx = 0
        for class_dir in sorted(self.root_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
            if len(images) < min_samples:
                continue
            self.class_to_idx[class_dir.name] = idx
            for img in images:
                self.samples.append((str(img), idx))
            idx += 1
        
        self.num_classes = len(self.class_to_idx)
        print(f"Loaded {len(self.samples)} images from {self.num_classes} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (160, 160))
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, label

print("Dataset class defined!")

## 4. Training Loop

In [ ]:
from facenet_pytorch import InceptionResnetV1
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import GradScaler, autocast
import time

# Config - adjusted for larger dataset
EPOCHS = 20  # Fewer epochs needed with 500K images
BATCH_SIZE = 128  # Larger batch for T4 GPU (16GB)
BACKBONE_LR = 1e-5
HEAD_LR = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Create datasets
train_dataset = FaceDataset('data/processed/train', transform=get_training_augmentation(), min_samples=3)
val_dataset = FaceDataset('data/processed/val', transform=get_validation_transform(), min_samples=1)

# Reduce batch size if dataset is small
if len(train_dataset) < 10000:
    BATCH_SIZE = 64
    EPOCHS = 30
    print(f"Small dataset detected, using batch_size={BATCH_SIZE}, epochs={EPOCHS}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Create model
model = InceptionResnetV1(pretrained='vggface2', classify=False, dropout_prob=0.6).to(device)
loss_fn = ArcFaceLoss(embedding_dim=512, num_classes=train_dataset.num_classes, scale=64.0, margin=0.5).to(device)

# Optimizer
optimizer = AdamW([
    {'params': model.parameters(), 'lr': BACKBONE_LR},
    {'params': loss_fn.parameters(), 'lr': HEAD_LR}
], weight_decay=5e-4)

scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-7)
scaler = GradScaler()

print(f"\nTraining config:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train samples: {len(train_dataset):,}")
print(f"  Val samples: {len(val_dataset):,}")
print(f"  Classes: {train_dataset.num_classes:,}")
print(f"  Batches per epoch: {len(train_loader):,}")

In [ ]:
# Training loop with progress tracking
best_accuracy = 0.0
LOG_INTERVAL = max(1, len(train_loader) // 10)  # Log 10 times per epoch

for epoch in range(EPOCHS):
    # Train
    model.train()
    loss_fn.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    start_time = time.time()
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        with autocast():
            embeddings = model(images)
            loss = loss_fn(embeddings, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(loss_fn.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        train_loss += loss.item()
        
        with torch.no_grad():
            cosine = loss_fn.get_cosine_similarity(F.normalize(embeddings, p=2, dim=1))
            pred = cosine.argmax(dim=1)
            train_correct += (pred == labels).sum().item()
            train_total += labels.size(0)
        
        # Progress log
        if (batch_idx + 1) % LOG_INTERVAL == 0:
            elapsed = time.time() - start_time
            samples_per_sec = train_total / elapsed
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Acc: {100*train_correct/train_total:.1f}% | "
                  f"Speed: {samples_per_sec:.0f} img/s")
    
    train_acc = train_correct / train_total
    
    # Validate
    model.eval()
    loss_fn.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            embeddings = model(images)
            cosine = loss_fn.get_cosine_similarity(F.normalize(embeddings, p=2, dim=1))
            pred = cosine.argmax(dim=1)
            val_correct += (pred == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    epoch_time = time.time() - start_time
    
    print(f"\nEpoch {epoch+1}/{EPOCHS} | Train: {train_acc:.2%} | Val: {val_acc:.2%} | Time: {epoch_time/60:.1f}min")
    
    # Save best model
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss_fn_state_dict': loss_fn.state_dict(),
            'val_accuracy': val_acc
        }, 'models/checkpoints/best_model.pth')
        print(f"  -> New best model! Accuracy: {val_acc:.2%}")
    
    scheduler.step()

print(f"\n{'='*50}")
print(f"Training complete! Best accuracy: {best_accuracy:.2%}")
print(f"{'='*50}")

## 5. Export Model

In [ ]:
# Load best model
checkpoint = torch.load('models/checkpoints/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Validation accuracy: {checkpoint['val_accuracy']:.2%}")

# Export to TorchScript
example_input = torch.randn(1, 3, 160, 160).to(device)
with torch.no_grad():
    traced_model = torch.jit.trace(model, example_input)

traced_model.save('models/final/heimdall_facenet_retrained.pt')
print("\nModel exported to: models/final/heimdall_facenet_retrained.pt")

# Also save state dict
torch.save(model.state_dict(), 'models/final/heimdall_facenet_retrained.pth')
print("State dict saved to: models/final/heimdall_facenet_retrained.pth")

## 6. Test on Noise

In [ ]:
# Quick noise test
import numpy as np

def add_noise(image, sigma):
    noise = np.random.normal(0, sigma, image.shape).astype(np.float32)
    noisy = np.clip(image.astype(np.float32) + noise, 0, 255)
    return noisy.astype(np.uint8)

# Test with validation set
test_dataset = FaceDataset('data/processed/test', transform=None, min_samples=1)

# Build gallery
gallery_dataset = FaceDataset('data/processed/train', transform=get_validation_transform(), min_samples=1)
gallery_loader = DataLoader(gallery_dataset, batch_size=64, shuffle=False)

gallery_embeddings = []
gallery_labels = []

model.eval()
with torch.no_grad():
    for images, labels in tqdm(gallery_loader, desc="Building gallery"):
        images = images.to(device)
        emb = model(images)
        emb = F.normalize(emb, p=2, dim=1)
        gallery_embeddings.append(emb.cpu().numpy())
        gallery_labels.append(labels.numpy())

gallery_embeddings = np.concatenate(gallery_embeddings)
gallery_labels = np.concatenate(gallery_labels)
print(f"Gallery size: {len(gallery_embeddings)}")

In [ ]:
# Test accuracy on different noise levels
noise_levels = [0, 10, 20, 30, 50]
results = {}

val_transform = get_validation_transform()

for sigma in noise_levels:
    correct = 0
    total = 0
    
    for idx in range(min(200, len(test_dataset))):
        img_path, label = test_dataset.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (160, 160))
        
        if sigma > 0:
            image = add_noise(image, sigma)
        
        augmented = val_transform(image=image)
        image_tensor = augmented['image'].unsqueeze(0).to(device)
        
        with torch.no_grad():
            embedding = model(image_tensor)
            embedding = F.normalize(embedding, p=2, dim=1).cpu().numpy()[0]
        
        # Match
        distances = 1 - np.dot(gallery_embeddings, embedding)
        min_idx = np.argmin(distances)
        predicted = gallery_labels[min_idx]
        
        if predicted == label:
            correct += 1
        total += 1
    
    accuracy = correct / total
    results[sigma] = accuracy
    print(f"Noise σ={sigma}: {accuracy:.2%} ({correct}/{total})")

print("\n" + "="*40)
print("RESULTS SUMMARY")
print("="*40)
for sigma, acc in results.items():
    label = "Original" if sigma == 0 else f"Noise σ={sigma}"
    print(f"{label}: {acc:.2%}")

## 7. Download Model

In [ ]:
# Download the trained model
from google.colab import files

print("Downloading trained model...")
files.download('models/final/heimdall_facenet_retrained.pt')

print("\nAfter downloading:")
print("1. Copy to backend/models/")
print("2. Run: python scripts/reencode_with_new_model.py --model models/heimdall_facenet_retrained.pt")